# Reti Neurali Feedforward

In questo notebook implementiamo una rete neurale feedforward, detta anche **Multi-Layer Perceptron**.

L'obiettivo è costruire un classificatore multiclasse partendo dai concetti base:

- input layer;
- hidden layer;
- output layer;
- forward propagation;
- funzione di costo;
- backpropagation;
- aggiornamento di pesi e bias.

La rete che implementeremo ha questa struttura:

```text
Input → Hidden layer → Output layer
```

Nel caso generale (con $t$ classi):
- ogni campione ha d feature;
- lo hidden layer contiene h neuroni;
- l'output layer contiene t neuroni, uno per ogni classe.

![schema rete](materiale_2026/15_01_nn.png)


---
## Architettura della rete

Consideriamo un singolo campione:

$$
\mathbf{x} = (x_1, x_2, \dots, x_d) \in \mathbb{R}^d
$$

Lo hidden layer contiene `h` neuroni.

Per ogni neurone nascosto `k` calcoliamo:

$$
z_k^{H} = \sum_{f=1}^{d} w_{f,k}^{H} x_f + b_k^{H}
$$

Poi applichiamo una funzione di attivazione:

$$
a_k^{H} = \sigma(z_k^{H})
$$

dove la sigmoid è:

$$
\sigma(z) = \frac{1}{1 + e^{-z}}
$$

L'output dello hidden layer è quindi:

$$
\mathbf{a}^{H} = (a_1^H, a_2^H, \dots, a_h^H)
$$

## Output layer

L'output layer riceve in input le attivazioni dello hidden layer.

Per ogni neurone di output `e`:

$$
z_e^{OUT} = \sum_{i=1}^{h} w_{i,e}^{OUT} a_i^H + b_e^{OUT}
$$

Poi applichiamo ancora la sigmoid:

$$
a_e^{OUT} = \sigma(z_e^{OUT})
$$

Se abbiamo `t` classi, l'output finale sarà un vettore:

$$
\mathbf{a}^{OUT} = (a_1^{OUT}, a_2^{OUT}, \dots, a_t^{OUT})
$$

Ogni componente rappresenta il livello di attivazione associato a una classe.

La classe predetta sarà quella con valore massimo:

$$
\hat{y} = \arg\max_e a_e^{OUT}
$$

## Notazione matriciale

Nel codice non lavoreremo su un singolo campione alla volta, ma su una matrice di esempi.

Sia:

$$
X \in \mathbb{R}^{n \times d}
$$

dove:

- `n` è il numero di esempi;
- `d` è il numero di feature.

Le matrici dei pesi sono:

$$
W^H \in \mathbb{R}^{d \times h}
$$

$$
W^{OUT} \in \mathbb{R}^{h \times t}
$$

I bias sono:

$$
b^H \in \mathbb{R}^{h}
$$

$$
b^{OUT} \in \mathbb{R}^{t}
$$

La forward propagation diventa:

$$
Z^H = XW^H + b^H
$$

$$
A^H = \sigma(Z^H)
$$

$$
Z^{OUT} = A^H W^{OUT} + b^{OUT}
$$

$$
A^{OUT} = \sigma(Z^{OUT})
$$

## Costo computazionale della forward propagation

Il costo computazionale principale della forward propagation deriva dai prodotti tra matrici.

Nel primo passaggio calcoliamo:

$$
Z^H = XW^H + b^H
$$

con:

$$
X \in \mathbb{R}^{n \times d}
$$

e:

$$
W^H \in \mathbb{R}^{d \times h}
$$

Il costo del prodotto matriciale è:

$$
O(n \cdot d \cdot h)
$$

Nel secondo passaggio calcoliamo:

$$
Z^{OUT} = A^H W^{OUT} + b^{OUT}
$$

con:

$$
A^H \in \mathbb{R}^{n \times h}
$$

e:

$$
W^{OUT} \in \mathbb{R}^{h \times t}
$$

Il costo è:

$$
O(n \cdot h \cdot t)
$$

Quindi il costo totale della forward propagation è:

$$
O(n \cdot d \cdot h + n \cdot h \cdot t)
$$

Se lavoriamo su mini-batch di dimensione `m`, il costo diventa:

$$
O(m \cdot d \cdot h + m \cdot h \cdot t)
$$

Questo significa che il costo aumenta al crescere di:

- numero di esempi;
- numero di feature;
- numero di neuroni hidden;
- numero di classi;
- numero di layer.

In [104]:
import numpy as np
import os

from sklearn.model_selection import train_test_split

## Funzione sigmoid

La rete della lezione usa la funzione **sigmoid** come funzione di attivazione.

La sigmoid è definita come:

$$
\sigma(z) = \frac{1}{1 + e^{-z}}
$$

Questa funzione trasforma ogni valore reale in un valore compreso tra 0 e 1.

Nel codice useremo `np.clip` per evitare problemi numerici quando `z` assume valori troppo grandi o troppo piccoli.

In [105]:
def sigmoid(z):
    """
    Calcola la funzione sigmoid.

    La sigmoid trasforma un valore reale in un valore compreso tra 0 e 1.

    Parametri
    ---------
    z : array-like
        Valore o matrice di valori in input.

    Ritorna
    -------
    array-like
        Sigmoid applicata elemento per elemento.
    """
    return 1.0 / (1.0 + np.exp(-np.clip(z, -250, 250)))#np.clip(z, -250, 250) serve a evitare overflow numerici quando z è molto grande o molto piccolo.

## Derivata della sigmoid

Durante la backpropagation serve la derivata della funzione di attivazione.

La derivata della sigmoid è:

$$
\sigma'(z) = \sigma(z)(1 - \sigma(z))
$$

Se indichiamo con:

$$
a = \sigma(z)
$$

allora possiamo scrivere:

$$
\sigma'(z) = a(1-a)
$$

Questa forma è comoda perché durante la forward propagation abbiamo già calcolato il valore dell'attivazione.

In [106]:
def sigmoid_derivative_from_activation(a):
    return a * (1.0 - a)

## One-hot encoding

Per un problema di classificazione multiclasse, le etichette vengono convertite in formato **one-hot**.

Esempio con tre classi:

Classe 0:

$$
[1, 0, 0]
$$

Classe 1:

$$
[0, 1, 0]
$$

Classe 2:

$$
[0, 0, 1]
$$

Questo formato è necessario perché l'output della rete contiene un valore per ogni classe.

Se il problema ha `t` classi, anche il target deve essere rappresentato come un vettore di lunghezza `t`.

In [107]:
def onehot(y, n_classes):
    onehot_matrix = np.zeros((y.shape[0], n_classes))

    for idx, class_label in enumerate(y.astype(int)):
        onehot_matrix[idx, class_label] = 1.0

    return onehot_matrix

In [108]:
y_demo = np.array([0, 1, 2, 1, 0])

y_demo_onehot = onehot(y_demo, n_classes=3)

print(y_demo_onehot)

[[1. 0. 0.]
 [0. 1. 0.]
 [0. 0. 1.]
 [0. 1. 0.]
 [1. 0. 0.]]


## Forward propagation

La **forward propagation** è la fase in cui la rete usa i pesi e i bias attuali per calcolare una predizione.

Nel nostro caso la rete ha due layer computazionali:

1. hidden layer;
2. output layer.

Il flusso è:

Input $X$ → Hidden layer → Output layer → Predizione

La forward propagation dello hidden layer è:

$$
Z^H = XW^H + b^H
$$

$$
A^H = \sigma(Z^H)
$$

La forward propagation dell'output layer è:

$$
Z^{OUT} = A^H W^{OUT} + b^{OUT}
$$

$$
A^{OUT} = \sigma(Z^{OUT})
$$

Dove:

- $X$ è la matrice degli input;
- $W^H$ sono i pesi tra input e hidden layer;
- $b^H$ sono i bias dello hidden layer;
- $A^H$ sono le attivazioni dello hidden layer;
- $W^{OUT}$ sono i pesi tra hidden layer e output layer;
- $b^{OUT}$ sono i bias dell'output layer;
- $A^{OUT}$ è l'output finale della rete.

Durante la forward propagation i pesi non vengono aggiornati: vengono solo usati per calcolare la predizione.

## Forward propagation dimostrativa

Prima di costruire la classe completa, implementiamo una forward propagation su dati casuali.

Supponiamo di avere:

- `n_examples = 5`;
- `n_features = 4`;
- `n_hidden = 3`;
- `n_classes = 2`.

Quindi:

$$
X \in \mathbb{R}^{5 \times 4}
$$

$$
W^H \in \mathbb{R}^{4 \times 3}
$$

$$
b^H \in \mathbb{R}^{3}
$$

$$
W^{OUT} \in \mathbb{R}^{3 \times 2}
$$

$$
b^{OUT} \in \mathbb{R}^{2}
$$

Il risultato finale della rete sarà:

$$
A^{OUT} \in \mathbb{R}^{5 \times 2}
$$

cioè una riga per ogni esempio e una colonna per ogni classe.

In [109]:
rng = np.random.RandomState(1)

n_examples = 5
n_features = 4
n_hidden = 3
n_classes = 2

X_demo = rng.normal(size=(n_examples, n_features))

W_h = rng.normal(loc=0.0, scale=0.1, size=(n_features, n_hidden))
b_h = np.zeros(n_hidden)

W_out = rng.normal(loc=0.0, scale=0.1, size=(n_hidden, n_classes))
b_out = np.zeros(n_classes)

print("X_demo shape:", X_demo.shape)
print("W_h shape:", W_h.shape)
print("b_h shape:", b_h.shape)
print("W_out shape:", W_out.shape)
print("b_out shape:", b_out.shape)

X_demo shape: (5, 4)
W_h shape: (4, 3)
b_h shape: (3,)
W_out shape: (3, 2)
b_out shape: (2,)


In [110]:
z_h = np.dot(X_demo, W_h) + b_h
a_h = sigmoid(z_h)

z_out = np.dot(a_h, W_out) + b_out
a_out = sigmoid(z_out)

print("z_h shape:", z_h.shape)
print("a_h shape:", a_h.shape)
print("z_out shape:", z_out.shape)
print("a_out shape:", a_out.shape)

print()
print("Output finale della rete:")
print(a_out)

z_h shape: (5, 3)
a_h shape: (5, 3)
z_out shape: (5, 2)
a_out shape: (5, 2)

Output finale della rete:
[[0.46744543 0.49390967]
 [0.4696473  0.49397548]
 [0.46914224 0.4932565 ]
 [0.46968395 0.49252688]
 [0.46952667 0.49222061]]


## Interpretazione dell'output

La matrice `a_out` contiene l'output finale della rete.

Nel nostro esempio:

$$
A^{OUT} \in \mathbb{R}^{5 \times 2}
$$

Questo significa che abbiamo:

- 5 esempi;
- 2 valori di output per ogni esempio.

Ogni riga rappresenta un esempio.

Ogni colonna rappresenta una classe.

Esempio:

$$
[0.48, 0.53]
$$

Il valore più alto si trova nella seconda posizione, quindi la rete assegna l'esempio alla classe `1`.

Per ottenere la classe predetta usiamo:

`np.argmax(a_out, axis=1)`

L'argomento `axis=1` indica che vogliamo prendere il massimo lungo le colonne, cioè separatamente per ogni esempio.

In [111]:
y_pred_demo = np.argmax(a_out, axis=1)

print("Classi predette:")
print(y_pred_demo)

Classi predette:
[1 1 1 1 1]


## Funzione di costo

La funzione di costo misura quanto la rete sta sbagliando.

Nel caso della lezione, l'output della rete è un vettore con un valore per ogni classe.

Per confrontare l'output della rete con la classe corretta, trasformiamo le etichette in formato one-hot.

Esempio:

Output della rete:

$$
A^{OUT} = [0.20, 0.70, 0.10]
$$

Target corretto:

$$
Y = [0, 1, 0]
$$

La rete dovrebbe:

- abbassare il valore della classe 0;
- alzare il valore della classe 1;
- abbassare il valore della classe 2.

La loss usata nella lezione è una somma di log loss binarie, una per ogni neurone di output:

$$
J = -\sum \left[Y \log(A^{OUT}) + (1-Y)\log(1-A^{OUT})\right]
$$

Questa funzione penalizza:

- la classe corretta se ha un valore troppo basso;
- le classi sbagliate se hanno un valore troppo alto.

## Regolarizzazione L2

Alla funzione di costo viene aggiunto un termine di regolarizzazione L2.

La regolarizzazione L2 penalizza pesi troppo grandi.

Il termine L2 è:

$$
L2 = \lambda \left(\sum (W^H)^2 + \sum (W^{OUT})^2\right)
$$

Dove:

- $\lambda$ è il coefficiente di regolarizzazione;
- $W^H$ sono i pesi tra input e hidden layer;
- $W^{OUT}$ sono i pesi tra hidden layer e output layer.

La funzione costo completa diventa:

$$
J = J_{data} + L2
$$

La regolarizzazione viene applicata ai pesi, non ai bias.

Questo aiuta a ridurre il rischio di overfitting, perché scoraggia il modello dal costruire pesi troppo grandi e troppo specifici sui dati di training.

In [112]:
def compute_cost(y_enc, output, W_h, W_out, l2=0.0):
    eps = 1e-15
    output = np.clip(output, eps, 1.0 - eps)

    term1 = -y_enc * np.log(output)
    term2 = -(1.0 - y_enc) * np.log(1.0 - output)

    data_loss = np.sum(term1 + term2)

    l2_term = l2 * (np.sum(W_h ** 2) + np.sum(W_out ** 2))

    cost = data_loss + l2_term

    return cost

## Esempio di calcolo della funzione costo

Costruiamo un piccolo esempio con 5 esempi e 2 classi.

La rete ha prodotto `a_out`, cioè una matrice di output con shape:

$$
5 \times 2
$$

Ora costruiamo delle etichette finte:

$$
y = [0, 1, 1, 0, 1]
$$

Poi le convertiamo in one-hot.

In questo modo possiamo confrontare:

$$
A^{OUT}
$$

con:

$$
Y_{onehot}
$$

e calcolare la loss.

In [113]:
y_demo = np.array([0, 1, 1, 0, 1])

y_demo_enc = onehot(y_demo, n_classes=2)

cost_demo = compute_cost(
    y_enc=y_demo_enc,
    output=a_out,
    W_h=W_h,
    W_out=W_out,
    l2=0.01
)

print("Target one-hot:")
print(y_demo_enc)

print()
print("Costo:")
print(cost_demo)

Target one-hot:
[[1. 0.]
 [0. 1.]
 [0. 1.]
 [1. 0.]
 [0. 1.]]

Costo:
6.898771630973499


## Costo computazionale della funzione costo

Il calcolo della loss confronta la matrice degli output con la matrice dei target one-hot.

Se il mini-batch contiene `m` esempi e il problema ha `t` classi, allora:

$$
A^{OUT} \in \mathbb{R}^{m \times t}
$$

e:

$$
Y \in \mathbb{R}^{m \times t}
$$

Il costo computazionale del confronto tra output e target è:

$$
O(m \cdot t)
$$

Il termine di regolarizzazione L2 richiede invece di sommare tutti i pesi della rete.

I pesi sono:

$$
W^H \in \mathbb{R}^{d \times h}
$$

e:

$$
W^{OUT} \in \mathbb{R}^{h \times t}
$$

Quindi il costo della regolarizzazione è:

$$
O(d \cdot h + h \cdot t)
$$

Il costo totale della funzione costo è:

$$
O(m \cdot t + d \cdot h + h \cdot t)
$$

Nella pratica, però, il costo principale dell'addestramento non è la loss, ma i prodotti matriciali della forward propagation e della backpropagation.

## Riassunto della prima parte

Fino a questo punto abbiamo implementato:

1. la funzione sigmoid;
2. la derivata della sigmoid;
3. la one-hot encoding;
4. la forward propagation;
5. la predizione tramite `argmax`;
6. la funzione costo;
7. la regolarizzazione L2;
8. l'analisi del costo computazionale della forward propagation e della loss.

Il prossimo passo è la **backpropagation**.

La backpropagation serve a calcolare i gradienti della funzione costo rispetto ai pesi e ai bias della rete.

Una volta calcolati i gradienti, possiamo aggiornare i parametri con gradient descent.

## Backpropagation

La **backpropagation** è l'algoritmo che permette alla rete di calcolare come modificare pesi e bias per ridurre la funzione costo.

Durante la forward propagation la rete calcola una predizione:

$$
A^{OUT}
$$

Durante la backpropagation confrontiamo questa predizione con il target corretto:

$$
Y
$$

e calcoliamo l'errore.

L'obiettivo è trovare i gradienti della funzione costo rispetto ai parametri della rete:

$$
\frac{\partial J}{\partial W^H}
$$

$$
\frac{\partial J}{\partial b^H}
$$

$$
\frac{\partial J}{\partial W^{OUT}}
$$

$$
\frac{\partial J}{\partial b^{OUT}}
$$

Questi gradienti indicano in quale direzione modificare pesi e bias per ridurre l'errore.

## Errore dell'output layer

Il primo passo della backpropagation è calcolare l'errore dello strato di output.

Nel caso della rete della lezione, l'errore dell'output layer è:

$$
\delta^{OUT} = A^{OUT} - Y
$$

Dove:

- $A^{OUT}$ è l'output prodotto dalla rete;
- $Y$ è il target in formato one-hot.

Esempio:

$$
A^{OUT} = [0.20, 0.70, 0.10]
$$

$$
Y = [0, 1, 0]
$$

Allora:

$$
\delta^{OUT} = [0.20, -0.30, 0.10]
$$

Interpretazione:

- la classe 0 ha output troppo alto, quindi deve essere abbassata;
- la classe 1 ha output troppo basso, quindi deve essere alzata;
- la classe 2 ha output troppo alto, quindi deve essere abbassata.

In [114]:
delta_out = a_out - y_demo_enc

print("a_out:")
print(a_out)

print()
print("y_demo_enc:")
print(y_demo_enc)

print()
print("delta_out:")
print(delta_out)

a_out:
[[0.46744543 0.49390967]
 [0.4696473  0.49397548]
 [0.46914224 0.4932565 ]
 [0.46968395 0.49252688]
 [0.46952667 0.49222061]]

y_demo_enc:
[[1. 0.]
 [0. 1.]
 [0. 1.]
 [1. 0.]
 [0. 1.]]

delta_out:
[[-0.53255457  0.49390967]
 [ 0.4696473  -0.50602452]
 [ 0.46914224 -0.5067435 ]
 [-0.53031605  0.49252688]
 [ 0.46952667 -0.50777939]]


## Errore dello hidden layer

Dopo aver calcolato l'errore dell'output layer, dobbiamo propagare questo errore verso lo hidden layer.

La formula è:

$$
\delta^H =
(\delta^{OUT}(W^{OUT})^T) \odot \sigma'(Z^H)
$$

Poiché:

$$
A^H = \sigma(Z^H)
$$

possiamo scrivere:

$$
\sigma'(Z^H) = A^H(1 - A^H)
$$

Quindi:

$$
\delta^H =
(\delta^{OUT}(W^{OUT})^T) \odot A^H(1 - A^H)
$$

Dove:

- $\delta^{OUT}$ è l'errore dell'output layer;
- $(W^{OUT})^T$ serve a riportare l'errore verso lo hidden layer;
- $A^H(1 - A^H)$ è la derivata della sigmoid nello hidden layer;
- $\odot$ indica il prodotto elemento per elemento.

In [115]:
sigmoid_derivative_h = sigmoid_derivative_from_activation(a_h)

delta_h = np.dot(delta_out, W_out.T) * sigmoid_derivative_h

print("sigmoid_derivative_h shape:", sigmoid_derivative_h.shape)
print("delta_h shape:", delta_h.shape)

print()
print("delta_h:")
print(delta_h)

sigmoid_derivative_h shape: (5, 3)
delta_h shape: (5, 3)

delta_h:
[[-0.00126595  0.00863988  0.01750621]
 [ 0.00257596 -0.00762925 -0.01589299]
 [ 0.00262788 -0.00771155 -0.0160428 ]
 [-0.00129568  0.00872088  0.01769911]
 [ 0.00266328 -0.00767906 -0.01608932]]


## Gradienti dei pesi tra hidden layer e output layer

Ora che conosciamo l'errore dell'output layer, possiamo calcolare il gradiente dei pesi tra hidden layer e output layer.

La formula è:

$$
\frac{\partial J}{\partial W^{OUT}} =
(A^H)^T \delta^{OUT}
$$

Nel codice:

```python
grad_w_out = np.dot(a_h.T, delta_out)
```

Intuizione:

- $A^H$ rappresenta ciò che lo hidden layer ha prodotto;
- $\delta^{OUT}$ rappresenta l'errore dell'output layer;
- il gradiente misura quanto ogni collegamento hidden → output ha contribuito all'errore.

Le dimensioni sono:

$$
(A^H)^T \in \mathbb{R}^{h \times m}
$$

$$
\delta^{OUT} \in \mathbb{R}^{m \times t}
$$

Quindi:

$$
grad\_w\_out \in \mathbb{R}^{h \times t}
$$

che è la stessa dimensione di:

$$
W^{OUT}
$$

In [116]:
grad_w_out = np.dot(a_h.T, delta_out)

print("a_h.T shape:", a_h.T.shape)
print("delta_out shape:", delta_out.shape)
print("grad_w_out shape:", grad_w_out.shape)
print("W_out shape:", W_out.shape)

print()
print("grad_w_out:")
print(grad_w_out)

a_h.T shape: (3, 5)
delta_out shape: (5, 2)
grad_w_out shape: (3, 2)
W_out shape: (3, 2)

grad_w_out:
[[ 0.16290579 -0.25015725]
 [ 0.11242173 -0.20507805]
 [ 0.17720619 -0.27699883]]


## Gradiente del bias dell'output layer

Il bias non viene moltiplicato per un input.

Per questo motivo il gradiente del bias è semplicemente la somma degli errori del layer.

Per l'output layer:

$$
\frac{\partial J}{\partial b^{OUT}} =
\sum \delta^{OUT}
$$

Nel codice:

```python
grad_b_out = np.sum(delta_out, axis=0)
```

L'argomento:

```python
axis=0
```

indica che stiamo sommando lungo le righe, cioè su tutti gli esempi del mini-batch.

Il risultato ha una componente per ogni neurone di output.

In [117]:
grad_b_out = np.sum(delta_out, axis=0)

print("delta_out shape:", delta_out.shape)
print("grad_b_out shape:", grad_b_out.shape)
print("b_out shape:", b_out.shape)

print()
print("grad_b_out:")
print(grad_b_out)

delta_out shape: (5, 2)
grad_b_out shape: (2,)
b_out shape: (2,)

grad_b_out:
[ 0.34544558 -0.53411086]


## Gradienti dei pesi tra input layer e hidden layer

Per calcolare i gradienti dei pesi tra input layer e hidden layer usiamo l'errore dello hidden layer:

$$
\delta^H
$$

La formula è:

$$
\frac{\partial J}{\partial W^H} =
X^T \delta^H
$$

Nel codice:

```python
grad_w_h = np.dot(X.T, delta_h)
```

Intuizione:

- $X$ rappresenta gli input ricevuti dallo hidden layer;
- $\delta^H$ rappresenta l'errore dei neuroni hidden;
- il gradiente misura quanto ogni collegamento input → hidden ha contribuito all'errore.

Le dimensioni sono:

$$
X^T \in \mathbb{R}^{d \times m}
$$

$$
\delta^H \in \mathbb{R}^{m \times h}
$$

Quindi:

$$
grad\_w\_h \in \mathbb{R}^{d \times h}
$$

che è la stessa dimensione di:

$$
W^H
$$

In [118]:
grad_w_h = np.dot(X_demo.T, delta_h)

print("X_demo.T shape:", X_demo.T.shape)
print("delta_h shape:", delta_h.shape)
print("grad_w_h shape:", grad_w_h.shape)
print("W_h shape:", W_h.shape)

print()
print("grad_w_h:")
print(grad_w_h)

X_demo.T shape: (4, 5)
delta_h shape: (5, 3)
grad_w_h shape: (4, 3)
W_h shape: (4, 3)

grad_w_h:
[[ 0.00096983  0.00348376  0.00663169]
 [-0.0076499   0.01758839  0.03719612]
 [ 0.00764887 -0.01958677 -0.04104536]
 [-0.003039   -0.00164348 -0.00247952]]


## Gradiente del bias dello hidden layer

Anche per lo hidden layer il bias viene aggiornato sommando gli errori del layer.

La formula è:

$$
\frac{\partial J}{\partial b^H} =
\sum \delta^H
$$

Nel codice:

```python
grad_b_h = np.sum(delta_h, axis=0)
```

Il risultato ha una componente per ogni neurone hidden.

In [119]:
grad_b_h = np.sum(delta_h, axis=0)

print("delta_h shape:", delta_h.shape)
print("grad_b_h shape:", grad_b_h.shape)
print("b_h shape:", b_h.shape)

print()
print("grad_b_h:")
print(grad_b_h)

delta_h shape: (5, 3)
grad_b_h shape: (3,)
b_h shape: (3,)

grad_b_h:
[ 0.00530549 -0.00565911 -0.01281979]


## Regolarizzazione L2 nei gradienti

Nella lezione viene usata anche la regolarizzazione L2.

La regolarizzazione L2 aggiunge una penalizzazione sui pesi troppo grandi.

Durante l'aggiornamento, ai gradienti dei pesi aggiungiamo:

$$
\lambda W
$$

Quindi:

$$
\Delta W^H = grad\_w\_h + \lambda W^H
$$

$$
\Delta W^{OUT} = grad\_w\_out + \lambda W^{OUT}
$$

I bias invece non vengono regolarizzati:

$$
\Delta b^H = grad\_b\_h
$$

$$
\Delta b^{OUT} = grad\_b\_out
$$

Quindi la regolarizzazione L2 agisce sui pesi, ma non sui bias.

In [120]:
l2 = 0.01

delta_w_h = grad_w_h + l2 * W_h
delta_b_h = grad_b_h

delta_w_out = grad_w_out + l2 * W_out
delta_b_out = grad_b_out

print("delta_w_h shape:", delta_w_h.shape)
print("delta_b_h shape:", delta_b_h.shape)
print("delta_w_out shape:", delta_w_out.shape)
print("delta_b_out shape:", delta_b_out.shape)

delta_w_h shape: (4, 3)
delta_b_h shape: (3,)
delta_w_out shape: (3, 2)
delta_b_out shape: (2,)


## Aggiornamento dei parametri

Una volta calcolati i gradienti, possiamo aggiornare pesi e bias.

La regola generale del gradient descent è:

$$
parametro := parametro - \eta \cdot gradiente
$$

Dove:

- $\eta$ è il learning rate;
- il gradiente indica la direzione in cui la funzione costo aumenta;
- sottraendo il gradiente ci muoviamo nella direzione in cui la funzione costo diminuisce.

Per la nostra rete:

$$
W^H := W^H - \eta \Delta W^H
$$

$$
b^H := b^H - \eta \Delta b^H
$$

$$
W^{OUT} := W^{OUT} - \eta \Delta W^{OUT}
$$

$$
b^{OUT} := b^{OUT} - \eta \Delta b^{OUT}
$$

In [121]:
eta = 0.01

W_h_updated = W_h - eta * delta_w_h
b_h_updated = b_h - eta * delta_b_h

W_out_updated = W_out - eta * delta_w_out
b_out_updated = b_out - eta * delta_b_out

print("W_h prima:")
print(W_h)

print()
print("W_h dopo:")
print(W_h_updated)

print()
print("b_h prima:")
print(b_h)

print()
print("b_h dopo:")
print(b_h_updated)

W_h prima:
[[-0.11006192  0.11447237  0.09015907]
 [ 0.05024943  0.09008559 -0.06837279]
 [-0.01228902 -0.09357694 -0.02678881]
 [ 0.05303555 -0.06916608 -0.03967535]]

W_h dopo:
[[-0.11006061  0.11442609  0.09008374]
 [ 0.05032091  0.0899007  -0.06873791]
 [-0.01236428 -0.09337172 -0.02637568]
 [ 0.05306063 -0.06914272 -0.03964659]]

b_h prima:
[0. 0. 0.]

b_h dopo:
[-5.30548917e-05  5.65911047e-05  1.28197926e-04]


## Costo computazionale della backpropagation

Il costo computazionale della backpropagation è dominato dai prodotti matriciali necessari per calcolare i gradienti.

Per l'output layer calcoliamo:

$$
grad\_w\_out = (A^H)^T \delta^{OUT}
$$

con:

$$
(A^H)^T \in \mathbb{R}^{h \times m}
$$

e:

$$
\delta^{OUT} \in \mathbb{R}^{m \times t}
$$

Il costo è:

$$
O(m \cdot h \cdot t)
$$

Per propagare l'errore allo hidden layer calcoliamo:

$$
\delta^{OUT}(W^{OUT})^T
$$

con:

$$
\delta^{OUT} \in \mathbb{R}^{m \times t}
$$

e:

$$
(W^{OUT})^T \in \mathbb{R}^{t \times h}
$$

Il costo è:

$$
O(m \cdot t \cdot h)
$$

Per i gradienti del primo layer calcoliamo:

$$
grad\_w\_h = X^T \delta^H
$$

con:

$$
X^T \in \mathbb{R}^{d \times m}
$$

e:

$$
\delta^H \in \mathbb{R}^{m \times h}
$$

Il costo è:

$$
O(m \cdot d \cdot h)
$$

Quindi il costo totale della backpropagation è circa:

$$
O(m \cdot h \cdot t + m \cdot t \cdot h + m \cdot d \cdot h)
$$

Semplificando:

$$
O(m \cdot h \cdot t + m \cdot d \cdot h)
$$

La backpropagation ha quindi un costo dello stesso ordine della forward propagation.

## Riassunto della backpropagation

La backpropagation segue questi passaggi:

1. Calcola l'errore dell'output layer:

$$
\delta^{OUT} = A^{OUT} - Y
$$

2. Propaga l'errore allo hidden layer:

$$
\delta^H =
(\delta^{OUT}(W^{OUT})^T) \odot A^H(1-A^H)
$$

3. Calcola i gradienti dei pesi hidden → output:

$$
grad\_w\_out = (A^H)^T \delta^{OUT}
$$

4. Calcola i gradienti dei bias output:

$$
grad\_b\_out = \sum \delta^{OUT}
$$

5. Calcola i gradienti dei pesi input → hidden:

$$
grad\_w\_h = X^T \delta^H
$$

6. Calcola i gradienti dei bias hidden:

$$
grad\_b\_h = \sum \delta^H
$$

7. Aggiorna pesi e bias:

$$
W := W - \eta \cdot grad_W
$$

$$
b := b - \eta \cdot grad_b
$$

Questi passaggi vengono ripetuti per ogni mini-batch e per ogni epoca.

## Forward propagation e backpropagation insieme

Il training di una rete neurale è quindi composto da due fasi principali.

Forward propagation:

$$
X \rightarrow Z^H \rightarrow A^H \rightarrow Z^{OUT} \rightarrow A^{OUT}
$$

Backpropagation:

$$
A^{OUT} - Y \rightarrow \delta^{OUT} \rightarrow \delta^H \rightarrow gradienti
$$

Aggiornamento:

$$
W := W - \eta \cdot grad_W
$$

$$
b := b - \eta \cdot grad_b
$$

Durante la forward propagation la rete usa i parametri attuali per calcolare una predizione.

Durante la backpropagation la rete calcola come modificare quei parametri.

Durante l'aggiornamento, pesi e bias vengono effettivamente cambiati.

In [122]:
class NeuralNetMLP(object):
    """ Feedforward neural network / Multi-layer perceptron classifier.

    Parameters
    ------------
    n_hidden : int (default: 30)
        Number of hidden units.
    epochs : int (default: 100)
        Number of passes over the training set.
    eta : float (default: 0.001)
        Learning rate.
    shuffle : bool (default: True)
        Shuffles training data every epoch if True to prevent circles.
    minibatch_size : int (default: 1)
        Number of training examples per minibatch.
    seed : int (default: None)
        Random seed for initializing weights and shuffling.

    Attributes
    -----------
    eval_ : dict
      Dictionary collecting the cost, training accuracy,
      and validation accuracy for each epoch during training.

    """
    def __init__(self, n_hidden=30, epochs=100, eta=0.001,
                 shuffle=True, minibatch_size=1, seed=None, l2 = 0):

        self.random = np.random.RandomState(seed)
        self.n_hidden = n_hidden
        self.epochs = epochs
        self.eta = eta
        self.shuffle = shuffle
        self.minibatch_size = minibatch_size
        self.l2 = l2

    def _onehot(self, y, n_classes):
        """Encode labels into one-hot representation

        Parameters
        ------------
        y : array, shape = [n_examples]
            Target values.
        n_classes : int
            Number of classes

        Returns
        -----------
        onehot : array, shape = (n_examples, n_labels)

        """
        onehot = np.zeros((n_classes, y.shape[0]))
        for idx, val in enumerate(y.astype(int)):
            onehot[val, idx] = 1.
        return onehot.T # <<<

    def _sigmoid(self, z):
        """Compute logistic function (sigmoid)"""
        return 1. / (1. + np.exp(-np.clip(z, -250, 250)))

    def _forward(self, X):
        """Compute forward propagation step"""

        # step 1: net input of hidden layer
        # [n_examples, n_features] dot [n_features, n_hidden]
        # -> [n_examples, n_hidden]
        z_h = np.dot(X, self.w_h) + self.b_h

        # step 2: activation of hidden layer
        a_h = self._sigmoid(z_h)

        # step 3: net input of output layer
        # [n_examples, n_hidden] dot [n_hidden, n_classlabels]
        # -> [n_examples, n_classlabels]

        z_out = np.dot(a_h, self.w_out) + self.b_out

        # step 4: activation output layer
        a_out = self._sigmoid(z_out)

        return z_h, a_h, z_out, a_out

    def _compute_cost(self, y_enc, output):
        """Compute cost function.

        Parameters
        ----------
        y_enc : array, shape = (n_examples, n_labels)
            one-hot encoded class labels.
        output : array, shape = [n_examples, n_output_units]
            Activation of the output layer (forward propagation)

        Returns
        ---------
        cost : float

        """

        L2_term = self.l2 *(np.sum(self.w_h**2) + np.sum(self.w_out**2))

        term1 = -y_enc * (np.log(output))
        term2 = (1. - y_enc) * np.log(1. - output)
        cost = np.sum(term1 - term2) + L2_term
        
        # If you are applying this cost function to other
        # datasets where activation
        # values maybe become more extreme (closer to zero or 1)
        # you may encounter "ZeroDivisionError"s due to numerical
        # instabilities in Python & NumPy for the current implementation.
        # I.e., the code tries to evaluate log(0), which is undefined.
        # To address this issue, you could add a small constant to the
        # activation values that are passed to the log function.
        #
        # For example:
        #
        # term1 = -y_enc * (np.log(output + 1e-5))
        # term2 = (1. - y_enc) * np.log(1. - output + 1e-5)
        
        return cost

    def predict(self, X):
        """Predict class labels

        Parameters
        -----------
        X : array, shape = [n_examples, n_features]
            Input layer with original features.

        Returns:
        ----------
        y_pred : array, shape = [n_examples]
            Predicted class labels.

        """
        z_h, a_h, z_out, a_out = self._forward(X)
        y_pred = np.argmax(z_out, axis=1)
        return y_pred

    def fit(self, X_train, y_train, X_valid, y_valid):
        """ Learn weights from training data.

        Parameters
        -----------
        X_train : array, shape = [n_examples, n_features]
            Input layer with original features.
        y_train : array, shape = [n_examples]
            Target class labels.
        X_valid : array, shape = [n_examples, n_features]
            Sample features for validation during training
        y_valid : array, shape = [n_examples]
            Sample labels for validation during training

        Returns:
        ----------
        self

        """
        n_output = np.unique(y_train).shape[0]  # number of class labels
        n_features = X_train.shape[1]

        ########################
        # Weight initialization
        ########################

        # weights for input -> hidden
        self.b_h = np.zeros(self.n_hidden)
        self.w_h = self.random.normal(loc=0.0, scale=0.1,
                                      size=(n_features, self.n_hidden))

        # weights for hidden -> output
        self.b_out = np.zeros(n_output)
        self.w_out = self.random.normal(loc=0.0, scale=0.1,
                                        size=(self.n_hidden, n_output))

        self.eval_ = {'cost': [], 'train_acc': [], 'valid_acc': []}

        y_train_enc = self._onehot(y_train, n_output)

        # iterate over training epochs
        for i in range(self.epochs):

            # iterate over minibatches
            indices = np.arange(X_train.shape[0])

            if self.shuffle:
                self.random.shuffle(indices)

            for start_idx in range(0, indices.shape[0] - self.minibatch_size +
                                   1, self.minibatch_size):
                batch_idx = indices[start_idx:start_idx + self.minibatch_size]

                # forward propagation
                z_h, a_h, z_out, a_out = self._forward(X_train[batch_idx])

                ##################
                # Backpropagation
                ##################

                # [n_examples, n_classlabels]

                delta_out = a_out - y_train_enc[batch_idx, :]

                # [n_examples, n_hidden]
                sigmoid_derivative_h = a_h * (1. - a_h)

                # [n_examples, n_classlabels] dot [n_classlabels, n_hidden]
                # -> [n_examples, n_hidden]
                delta_h = (np.dot(delta_out, self.w_out.T) *
                           sigmoid_derivative_h)

                # [n_features, n_examples] dot [n_examples, n_hidden]
                # -> [n_features, n_hidden]
                grad_w_h = np.dot(X_train[batch_idx].T, delta_h)
                grad_b_h = np.sum(delta_h, axis=0)
                # O(d*n*h)
                
                # [n_hidden, n_examples] dot [n_examples, n_classlabels]
                # -> [n_hidden, n_classlabels]
                grad_w_out = np.dot(a_h.T, delta_out)
                grad_b_out = np.sum(delta_out, axis=0)
                # O(h*n*t)

                # Regularization and weight updates
                delta_w_h = grad_w_h + self.l2*self.w_h
                delta_b_h = grad_b_h # bias is not regularized
                self.w_h -= self.eta * delta_w_h
                self.b_h -= self.eta * delta_b_h

                delta_w_out = grad_w_out + self.l2*self.w_out
                delta_b_out = grad_b_out  # bias is not regularized
                self.w_out -= self.eta * delta_w_out
                self.b_out -= self.eta * delta_b_out

                # O(n*h*(d+t)
                
            #############
            # Evaluation
            #############

            # Evaluation after each epoch during training
            z_h, a_h, z_out, a_out = self._forward(X_train)
            
            cost = self._compute_cost(y_enc=y_train_enc,
                                      output=a_out)

            y_train_pred = self.predict(X_train)
            y_valid_pred = self.predict(X_valid)

            train_acc = ((np.sum(y_train == y_train_pred)).astype(np.float64) /
                         X_train.shape[0])
            valid_acc = ((np.sum(y_valid == y_valid_pred)).astype(np.float64) /
                         X_valid.shape[0])

            if i%100 == 0:
                print(f"Cost: {cost} | Train/Valid Acc.: {train_acc*100}/{valid_acc*100}")

            self.eval_['cost'].append(cost)
            self.eval_['train_acc'].append(train_acc)
            self.eval_['valid_acc'].append(valid_acc)

        return self

In [123]:
X = np.genfromtxt(os.path.join('dataset','01-02-iris.csv'), delimiter=',', skip_header=0, usecols=(0, 1, 2, 3))

y = np.genfromtxt(os.path.join('dataset','01-02-iris.csv'), delimiter=',', skip_header=0, usecols=(4), dtype=str)

label_map = {'Iris-setosa':0, 'Iris-versicolor':1, 'Iris-virginica':2}
y = np.array([label_map[x] for x in y])


X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3)

In [124]:
nn = NeuralNetMLP(n_hidden=3,
                  epochs=2000, 
                  eta=0.0025,
                  minibatch_size=15, 
                  shuffle=True,
                  seed=1,
                  l2=0.01)
nn.fit(X_train=X_train, 
       y_train=y_train,
       X_valid=X_test,
       y_valid=y_test)

Cost: 213.08303922519528 | Train/Valid Acc.: 31.428571428571427/37.77777777777778
Cost: 129.9026389800636 | Train/Valid Acc.: 81.9047619047619/80.0
Cost: 104.92951558997125 | Train/Valid Acc.: 97.14285714285714/97.77777777777777
Cost: 90.15135151440344 | Train/Valid Acc.: 97.14285714285714/97.77777777777777
Cost: 72.49218972612806 | Train/Valid Acc.: 98.09523809523809/97.77777777777777
Cost: 58.460182167094864 | Train/Valid Acc.: 98.09523809523809/100.0
Cost: 48.22741017204251 | Train/Valid Acc.: 99.04761904761905/97.77777777777777
Cost: 41.63750145697265 | Train/Valid Acc.: 97.14285714285714/97.77777777777777
Cost: 36.674639914662485 | Train/Valid Acc.: 97.14285714285714/97.77777777777777
Cost: 33.46671348645042 | Train/Valid Acc.: 98.09523809523809/97.77777777777777
Cost: 31.21994655464559 | Train/Valid Acc.: 97.14285714285714/97.77777777777777
Cost: 29.4154255984606 | Train/Valid Acc.: 99.04761904761905/97.77777777777777
Cost: 29.28890345525066 | Train/Valid Acc.: 97.14285714285714/

Sperimentare il precedente codice usando questo dataset

In [142]:
from sklearn.datasets import make_classification
from sklearn.preprocessing import StandardScaler

X,y = make_classification(n_samples=1000, n_features=10,
                        n_informative=7, n_classes=4, random_state=0)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3)
scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)
nn = NeuralNetMLP(n_hidden=20,
                  epochs=6000, 
                  eta=0.005,
                  minibatch_size=25, 
                  shuffle=True,
                  seed=2,
                  l2=0.05)
nn.fit(X_train=X_train, 
       y_train=y_train,
       X_valid=X_test,
       y_valid=y_test)

Cost: 1554.5737611138647 | Train/Valid Acc.: 33.714285714285715/32.0
Cost: 1011.1193738500336 | Train/Valid Acc.: 68.0/59.66666666666667
Cost: 834.8007805495898 | Train/Valid Acc.: 78.42857142857143/69.33333333333334
Cost: 754.1436026323756 | Train/Valid Acc.: 80.71428571428572/73.0
Cost: 715.9360517821524 | Train/Valid Acc.: 81.71428571428572/75.0
Cost: 689.1750195612477 | Train/Valid Acc.: 83.71428571428572/75.0
Cost: 670.5297090052275 | Train/Valid Acc.: 85.14285714285714/75.33333333333333
Cost: 655.2160575538335 | Train/Valid Acc.: 85.57142857142857/76.66666666666667
Cost: 643.2079296539229 | Train/Valid Acc.: 85.85714285714286/76.0
Cost: 635.628962502654 | Train/Valid Acc.: 86.71428571428571/78.0
Cost: 631.1775844680255 | Train/Valid Acc.: 86.71428571428571/79.0
Cost: 628.928929277287 | Train/Valid Acc.: 87.42857142857143/78.66666666666666
Cost: 627.9963512197294 | Train/Valid Acc.: 86.85714285714286/78.66666666666666
Cost: 626.9465190525347 | Train/Valid Acc.: 87.28571428571429/7

## Effetto della standardizzazione

Abbiamo confrontato il training della rete con e senza standardizzazione delle feature.

Senza standardizzazione, la rete raggiunge una accuracy più alta sul training set, ma la validation accuracy rimane circa invariata.

Con standardizzazione, la training accuracy si abbassa, mentre la validation accuracy resta simile.

Questo indica che il modello non standardizzato riesce ad adattarsi meglio al training set, ma questa maggiore capacità non si traduce in una migliore generalizzazione.

In altre parole, la standardizzazione riduce il gap tra training e validation accuracy, producendo un modello più stabile e meno soggetto a overfitting.

La standardizzazione è particolarmente importante nelle reti con sigmoid, perché input con scale molto diverse possono portare la sigmoid a saturare, rendendo il training meno controllato.